In [72]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime, time
import re

# --- CONFIGURATION ---
BASE_PATH_SPOT = r"C:\Users\RISHI\Downloads\nif1\nifty_data\nifty_spot\2024"
BASE_PATH_OPT = r"C:\Users\RISHI\Downloads\nif1\nifty_data\nifty_options\2024"
LOT_SIZE = 50
SL_TICKS = 41 
TARGET_PTS = 99
ANGLE_THRESHOLD = 30 
TRADING_START_TIME = time(9, 30)
SQUARE_OFF_TIME = time(15, 15)
RSI_PERIOD = 14  

def get_atm_strike(spot_price):
    return int(round(spot_price / 50) * 50)

def parse_expiry_from_sym(symbol):
    try:
        match = re.search(r'(\d{2})([A-Z]{3})(\d{2})', symbol)
        if not match: return None
        p1, month, p2 = match.groups()
        try:
            dt = datetime.strptime(f"{p1}{month}{p2}", '%d%b%y').date()
            if dt.year >= 2020: return dt
        except: pass
        try:
            dt = datetime.strptime(f"{p1}{month}{p2}", '%y%b%d').date()
            if dt.year >= 2020: return dt
        except: pass
    except: return None
    return None

def calculate_indicators(df):
    df = df.copy()
    df['ema5'] = df['close'].ewm(span=5, adjust=False).mean()
    df['ema8'] = df['close'].ewm(span=8, adjust=False).mean()
    df['vwap'] = (df['close'] * df['volume']).cumsum() / df['volume'].cumsum()
    df['ema5_angle'] = np.degrees(np.arctan(df['ema5'].diff()))
    df['ema8_angle'] = np.degrees(np.arctan(df['ema8'].diff()))
    
    # RSI Calculation
    delta = df['close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/RSI_PERIOD, adjust=True, min_periods=1).mean()
    avg_loss = loss.ewm(alpha=1/RSI_PERIOD, adjust=True, min_periods=1).mean()
    rs = np.divide(avg_gain, avg_loss, out=np.full_like(avg_gain, np.nan), where=avg_loss != 0)
    df['rsi'] = 100 - (100 / (1 + rs))
    df.loc[avg_loss == 0, 'rsi'] = 100
    df.loc[(avg_loss == 0) & (avg_gain == 0), 'rsi'] = 50
    return df

def run_backtest():
    all_trades = []
    
    for month in range(1, 12):
        spot_folder = os.path.join(BASE_PATH_SPOT, str(month))
        opt_folder = os.path.join(BASE_PATH_OPT, str(month))
        if not os.path.exists(spot_folder): continue
        
        spot_files = sorted(glob.glob(os.path.join(spot_folder, "*.csv")))
        for s_file in spot_files:
            try:
                df_spot = pd.read_csv(s_file)
                df_spot['datetime'] = pd.to_datetime(df_spot['date'] + ' ' + df_spot['time'])
                current_date_obj = df_spot['datetime'].iloc[0].date()
                current_date_str = str(current_date_obj)
                
                spot_row = df_spot[df_spot['time'] == '09:16:00']
                if spot_row.empty: continue
                atm_strike = get_atm_strike(spot_row['close'].values[0])
                
                date_suffix = current_date_obj.strftime('%d_%m_%Y')
                opt_pattern = os.path.join(opt_folder, f"*{date_suffix}*.csv")
                matched_opt_files = glob.glob(opt_pattern)
                if not matched_opt_files: continue
                
                df_opt_full = pd.read_csv(matched_opt_files[0])
                df_opt_day = df_opt_full[df_opt_full['date'] == current_date_str].copy()
                if df_opt_day.empty: continue

                all_symbols = df_opt_day['symbol'].unique()

                for opt_type in ['CE', 'PE']:
                    strike_str = str(atm_strike)
                    matches = [s for s in all_symbols if (strike_str in s and opt_type in s)]
                    valid_expiries = []
                    for s in matches:
                        exp = parse_expiry_from_sym(s)
                        if exp and exp >= current_date_obj:
                            valid_expiries.append((s, exp))
                    
                    if not valid_expiries: continue
                    valid_expiries.sort(key=lambda x: x[1])
                    selected_sym = valid_expiries[0][0]
                    
                    contract_data = df_opt_day[df_opt_day['symbol'] == selected_sym].copy()
                    contract_data['datetime'] = pd.to_datetime(contract_data['date'] + ' ' + contract_data['time'])
                    contract_data.set_index('datetime', inplace=True)
                    contract_data.sort_index(inplace=True)
                    
                    df_3m = contract_data.resample('3min').agg({
                        'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'
                    }).dropna()
                    df_3m = calculate_indicators(df_3m)
                    
                    active_trade = None
                    for i in range(1, len(df_3m)):
                        curr = df_3m.iloc[i]
                        prev = df_3m.iloc[i-1]
                        curr_t = curr.name.time()
                        
                        # Time Exit
                        if curr_t >= SQUARE_OFF_TIME and active_trade is not None:
                            pnl = (curr['close'] - active_trade['Entry Price']) * LOT_SIZE
                            all_trades.append({
                                'Date': current_date_str, 'Symbol': selected_sym, 
                                'Entry Time': active_trade['Entry Time'], 'Exit Time': curr.name.strftime('%H:%M:%S'),
                                'Entry': active_trade['Entry Price'], 'Exit': curr['close'], 'PnL': pnl, 'Result': 'Time Exit'
                            })
                            active_trade = None; break

                        if active_trade is None:
                            if TRADING_START_TIME <= curr_t < SQUARE_OFF_TIME:
                                if (prev['ema5'] <= prev['ema8'] and curr['ema5'] > curr['ema8'] and 
                                    curr['close'] > curr['vwap'] and 
                                    curr['ema5_angle'] >= ANGLE_THRESHOLD and 
                                    curr['ema8_angle'] >= ANGLE_THRESHOLD and 
                                    curr['ema5'] > curr['vwap'] and
                                    curr['ema8'] > curr['vwap'] and
                                    curr['rsi'] < 80):
                                    
                                    active_trade = {
                                        'Entry Price': curr['close'], 
                                        'Entry Time': curr.name.strftime('%H:%M:%S'),
                                        'SL': curr['close'] - SL_TICKS, 
                                        'Target': curr['close'] + TARGET_PTS
                                    }
                        else:
                            profit = curr['close'] - active_trade['Entry Price']
                            steps = int(profit // 1)
                            if steps >= 1:
                                new_sl = (active_trade['Entry Price'] - SL_TICKS) + (steps * 0.5)
                                active_trade['SL'] = max(active_trade['SL'], new_sl)
                            
                            if curr['high'] >= active_trade['Target']:
                                all_trades.append({
                                    'Date': current_date_str, 'Symbol': selected_sym, 
                                    'Entry Time': active_trade['Entry Time'], 'Exit Time': curr.name.strftime('%H:%M:%S'),
                                    'Entry': active_trade['Entry Price'], 'Exit': active_trade['Target'], 'PnL': TARGET_PTS * LOT_SIZE, 'Result': 'Target'
                                })
                                active_trade = None; break
                            elif curr['low'] <= active_trade['SL']:
                                exit_val = active_trade['SL']
                                pnl = (exit_val - active_trade['Entry Price']) * LOT_SIZE
                                all_trades.append({
                                    'Date': current_date_str, 'Symbol': selected_sym, 
                                    'Entry Time': active_trade['Entry Time'], 'Exit Time': curr.name.strftime('%H:%M:%S'),
                                    'Entry': active_trade['Entry Price'], 'Exit': exit_val, 'PnL': pnl, 'Result': 'SL'
                                })
                                active_trade = None; break
            except Exception as e:
                print(f"Error on {s_file}: {e}")

    if all_trades:
        pd.DataFrame(all_trades).to_excel("Nifty_Backtest_Final_Results.xlsx", index=False)
        print("Backtest Finished. File: Nifty_Backtest_Final_Results.xlsx")
    else:
        print("No trades found.")

if __name__ == "__main__":
    run_backtest()

C:\Users\RISHI\AppData\Local\Temp\ipykernel_13236\1548536810.py:138: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vx = vix_3m.get(i)
C:\Users\RISHI\AppData\Local\Temp\ipykernel_13236\1548536810.py:138: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vx = vix_3m.get(i)
C:\Users\RISHI\AppData\Local\Temp\ipykernel_13236\1548536810.py:138: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vx = vix_3m.get(i)
C:\Users\RISHI\AppData\Local\Temp\

Backtest Finished. File: Nifty_Backtest_Final_Results.xlsx


C:\Users\RISHI\AppData\Local\Temp\ipykernel_13236\1548536810.py:138: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vx = vix_3m.get(i)
C:\Users\RISHI\AppData\Local\Temp\ipykernel_13236\1548536810.py:138: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vx = vix_3m.get(i)


In [2]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime, time
import re

# --- CONFIGURATION ---
BASE_PATH_SPOT = r"C:\Users\RISHI\Downloads\nif1\nifty_data\nifty_spot\2024"
BASE_PATH_OPT = r"C:\Users\RISHI\Downloads\nif1\nifty_data\nifty_options\2024"
VIX_FILE_PATH = "INDIA VIX_minute.csv"
LOT_SIZE = 50
SL_TICKS = 41 
TARGET_PTS = 99
ANGLE_THRESHOLD = 30 
TRADING_START_TIME = time(9, 30)
SQUARE_OFF_TIME = time(15, 15)
RSI_PERIOD = 14  

def get_atm_strike(spot_price):
    return int(round(spot_price / 50) * 50)

def parse_expiry_from_sym(symbol):
    try:
        match = re.search(r'(\d{2})([A-Z]{3})(\d{2})', symbol)
        if not match: return None
        p1, month, p2 = match.groups()
        try:
            dt = datetime.strptime(f"{p1}{month}{p2}", '%d%b%y').date()
            if dt.year >= 2020: return dt
        except: pass
        try:
            dt = datetime.strptime(f"{p1}{month}{p2}", '%y%b%d').date()
            if dt.year >= 2020: return dt
        except: pass
    except: return None
    return None

def calculate_indicators(df):
    df = df.copy()
    df['ema5'] = df['close'].ewm(span=5, adjust=False).mean()
    df['ema8'] = df['close'].ewm(span=8, adjust=False).mean()
    df['vwap'] = (df['close'] * df['volume']).cumsum() / df['volume'].cumsum()
    df['ema5_angle'] = np.degrees(np.arctan(df['ema5'].diff()))
    df['ema8_angle'] = np.degrees(np.arctan(df['ema8'].diff()))
    #df['ema5_angle'] = df['ema5'].pct_change()
    #df['ema8_angle'] = df['ema8'].pct_change()
    
    delta = df['close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/RSI_PERIOD, adjust=True, min_periods=1).mean()
    avg_loss = loss.ewm(alpha=1/RSI_PERIOD, adjust=True, min_periods=1).mean()
    rs = np.divide(avg_gain, avg_loss, out=np.full_like(avg_gain, np.nan), where=avg_loss != 0)
    df['rsi'] = 100 - (100 / (1 + rs))
    df.loc[avg_loss == 0, 'rsi'] = 100
    df.loc[(avg_loss == 0) & (avg_gain == 0), 'rsi'] = 50
    return df

def run_backtest():
    # 1. LOAD VIX DATA WITH FLEXIBLE COLUMN DETECTION
    vix_series = pd.Series(dtype=float)
    try:
        dv = pd.read_csv(VIX_FILE_PATH)
        dv.columns = [c.lower() for c in dv.columns] # Normalize column names
        
        if 'date' in dv.columns and 'time' in dv.columns:
            dv['datetime'] = pd.to_datetime(dv['date'] + ' ' + dv['time'])
        elif 'date' in dv.columns:
            dv['datetime'] = pd.to_datetime(dv['date'])
            
        vix_series = dv.set_index('datetime')['close'].sort_index()
        vix_series = vix_series.resample('1min').ffill()
    except Exception as e:
        print(f"Warning: INDIA VIX file error: {e}. VIX filter will be ignored.")

    all_trades = []
    
    for month in range(1, 13):
        spot_folder = os.path.join(BASE_PATH_SPOT, str(month))
        opt_folder = os.path.join(BASE_PATH_OPT, str(month))
        if not os.path.exists(spot_folder): continue
        
        spot_files = sorted(glob.glob(os.path.join(spot_folder, "*.csv")))
        for s_file in spot_files:
            try:
                df_spot = pd.read_csv(s_file)
                df_spot['datetime'] = pd.to_datetime(df_spot['date'] + ' ' + df_spot['time'])
                current_date_obj = df_spot['datetime'].iloc[0].date()
                current_date_str = str(current_date_obj)
                
                spot_row = df_spot[df_spot['time'] == '09:16:00']
                if spot_row.empty: continue
                atm_strike = get_atm_strike(spot_row['close'].values[0])
                
                date_suffix = current_date_obj.strftime('%d_%m_%Y')
                opt_pattern = os.path.join(opt_folder, f"*{date_suffix}*.csv")
                matched_opt_files = glob.glob(opt_pattern)
                if not matched_opt_files: continue
                
                df_opt_full = pd.read_csv(matched_opt_files[0])
                df_opt_day = df_opt_full[df_opt_full['date'] == current_date_str].copy()
                if df_opt_day.empty: continue

                all_symbols = df_opt_day['symbol'].unique()

                for opt_type in ['CE', 'PE']:
                    strike_str = str(atm_strike)
                    matches = [s for s in all_symbols if (strike_str in s and opt_type in s)]
                    valid_expiries = []
                    for s in matches:
                        exp = parse_expiry_from_sym(s)
                        if exp and exp >= current_date_obj:
                            valid_expiries.append((s, exp))
                    
                    if not valid_expiries: continue
                    valid_expiries.sort(key=lambda x: x[1])
                    selected_sym = valid_expiries[0][0]
                    
                    contract_data = df_opt_day[df_opt_day['symbol'] == selected_sym].copy()
                    contract_data['datetime'] = pd.to_datetime(contract_data['date'] + ' ' + contract_data['time'])
                    contract_data.set_index('datetime', inplace=True)
                    contract_data.sort_index(inplace=True)
                    
                    df_3m = contract_data.resample('3min').agg({
                        'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'
                    }).dropna()
                    df_3m = calculate_indicators(df_3m)
                    
                    active_trade = None
                    for i in range(1, len(df_3m)):
                        curr = df_3m.iloc[i]
                        prev = df_3m.iloc[i-1]
                        curr_t = curr.name.time()
                        
                        if curr_t >= SQUARE_OFF_TIME and active_trade is not None:
                            pnl = (curr['close'] - active_trade['Entry Price']) * LOT_SIZE
                            all_trades.append({
                                'Date': current_date_str, 'Symbol': selected_sym, 
                                'Entry Time': active_trade['Entry Time'], 'Exit Time': curr.name.strftime('%H:%M:%S'),
                                'Entry': active_trade['Entry Price'], 'Exit': curr['close'], 'PnL': pnl, 'Result': 'Time Exit'
                            })
                            active_trade = None; break

                        if active_trade is None:
                            if TRADING_START_TIME <= curr_t < SQUARE_OFF_TIME:
                                # Get VIX for current timestamp
                                vx = vix_series.get(curr.name) # Default 16 if VIX fails
                                
                                if (prev['ema5'] <= prev['ema8'] and curr['ema5'] > curr['ema8'] and 
                                    curr['close'] > curr['vwap'] and 
                                    curr['ema5_angle'] >= ANGLE_THRESHOLD and 
                                    curr['ema8_angle'] >= ANGLE_THRESHOLD and 
                                    curr['ema5'] > curr['vwap'] and
                                    curr['ema8'] > curr['vwap'] and
                                    curr['rsi'] < 80 and
                                    vx > 11 and vx < 21):
                                    
                                    active_trade = {
                                        'Entry Price': curr['close'], 
                                        'Entry Time': curr.name.strftime('%H:%M:%S'),
                                        'SL': curr['close'] - SL_TICKS, 
                                        'Target': curr['close'] + TARGET_PTS
                                    }
                        else:
                            profit = curr['close'] - active_trade['Entry Price']
                            steps = int(profit // 1)
                            if steps >= 1:
                                new_sl = (active_trade['Entry Price'] - SL_TICKS) + (steps * 0.5)
                                active_trade['SL'] = max(active_trade['SL'], new_sl)
                            
                            if curr['high'] >= active_trade['Target']:
                                all_trades.append({
                                    'Date': current_date_str, 'Symbol': selected_sym, 
                                    'Entry Time': active_trade['Entry Time'], 'Exit Time': curr.name.strftime('%H:%M:%S'),
                                    'Entry': active_trade['Entry Price'], 'Exit': active_trade['Target'], 'PnL': TARGET_PTS * LOT_SIZE, 'Result': 'Target'
                                })
                                active_trade = None; break
                            elif curr['low'] <= active_trade['SL']:
                                exit_val = active_trade['SL']
                                pnl = (exit_val - active_trade['Entry Price']) * LOT_SIZE
                                all_trades.append({
                                    'Date': current_date_str, 'Symbol': selected_sym, 
                                    'Entry Time': active_trade['Entry Time'], 'Exit Time': curr.name.strftime('%H:%M:%S'),
                                    'Entry': active_trade['Entry Price'], 'Exit': exit_val, 'PnL': pnl, 'Result': 'SL'
                                })
                                active_trade = None; break
            except Exception as e:
                print(f"Error on {s_file}: {e}")

    if all_trades:
        pd.DataFrame(all_trades).to_excel("Nifty_Backtest_Final_Results.xlsx", index=False)
        print("Backtest Finished. File: Nifty_Backtest_Final_Results.xlsx")
    else:
        print("No trades found. Check VIX threshold or date formats.")

if __name__ == "__main__":
    run_backtest()

Backtest Finished. File: Nifty_Backtest_Final_Results.xlsx


In [3]:
import pandas as pd

# Load the backtest results file
file_path = r'Nifty_Backtest_Final_Results.xlsx'
df = pd.read_excel(file_path)

# Convert PnL to numeric if it's not
df['PnL'] = pd.to_numeric(df['PnL'], errors='coerce')

# Calculate Summary
total_profit = df['PnL'].sum()
no_of_trades = len(df)
wins = len(df[df['PnL'] >= 0])
losses = len(df[df['PnL'] < 0])
win_rate = (wins / no_of_trades) * 100

# Calculate Drawdown
df['CumPnL'] = df['PnL'].cumsum()
df['MaxCum'] = df['CumPnL'].cummax()
df['Drawdown'] = df['MaxCum'] - df['CumPnL']
max_drawdown = df['Drawdown'].max()

# Sample of trades
sample_trades = df[['Date', 'Symbol', 'PnL']].head(10)

print(f"Total Profit: {total_profit}")
print(f"Total Trades: {no_of_trades}")
print(f"Max Drawdown: {max_drawdown}")
print("\nSample Trades:")
print(sample_trades)
print(win_rate)

Total Profit: 94720.0
Total Trades: 217
Max Drawdown: 16750.0

Sample Trades:
         Date               Symbol     PnL
0  2024-01-01  NIFTY04JAN2421700CE  4950.0
1  2024-01-02  NIFTY04JAN2421750PE  4950.0
2  2024-01-03  NIFTY04JAN2421600PE  2207.5
3  2024-01-04  NIFTY04JAN2421600CE    60.0
4  2024-01-08  NIFTY11JAN2421750PE  4950.0
5  2024-01-09  NIFTY11JAN2421650CE -1700.0
6  2024-01-09  NIFTY11JAN2421650PE -1900.0
7  2024-01-10  NIFTY11JAN2421550CE -2050.0
8  2024-01-10  NIFTY11JAN2421550PE -1925.0
9  2024-01-11  NIFTY11JAN2421700PE   665.0
39.1705069124424


In [40]:
pd.set_option('display.max_rows', None)
df

,Date,Symbol,Entry Time,Exit Time,Entry,Exit,PnL,Result,CumPnL,MaxCum,Drawdown
0,2024-01-01,NIFTY04JAN2421700CE,09:45:00,14:42:00,142.00,241.00,4950.0,Target,4950.0,4950.0,0.0
1,2024-01-02,NIFTY04JAN2421750PE,09:30:00,10:39:00,106.15,205.15,4950.0,Target,9900.0,9900.0,0.0
2,2024-01-03,NIFTY04JAN2421600PE,10:00:00,15:15:00,88.60,132.75,2207.5,Time Exit,12107.5,12107.5,0.0
3,2024-01-04,NIFTY04JAN2421600CE,11:39:00,15:15:00,51.40,52.60,60.0,Time Exit,12167.5,12167.5,0.0
4,2024-01-08,NIFTY11JAN2421750PE,09:30:00,11:54:00,116.45,215.45,4950.0,Target,17117.5,17117.5,0.0
5,2024-01-09,NIFTY11JAN2421650CE,11:30:00,14:06:00,125.30,91.30,-1700.0,SL,15417.5,17117.5,1700.0
6,2024-01-09,NIFTY11JAN2421650PE,09:39:00,10:42:00,93.60,55.60,-1900.0,SL,13517.5,17117.5,3600.0
7,2024-01-10,NIFTY11JAN2421550CE,09:42:00,12:45:00,87.25,46.25,-2050.0,SL,11467.5,17117.5,5650.0
8,2024-01-10,NIFTY11JAN2421550PE,13:18:00,14:00:00,107.45,68.95,-1925.0,SL,9542.5,17117.5,7575.0
9,2024-01-11,NIFTY11JAN2421700PE,09:30:00,15:15:00,40.65,53.95,665.0,Time Exit,10207.5,17117.5,6910.0
